# Deep Learning

End-to-end deep learning pipeline for 3-class N-Back EEG.

This notebook loads preprocessed epochs, normalizes them using baseline segments, explores the data,
trains models with cross-validation, and then trains a final model on all prepared data.
We also check generalization to the opposite session and analyze channel importance via occlusion.
Finally, we include an additional simple phase-based train/test split experiment and compute channel
importance for that setup as well.

In [ ]:
import mne
from pathlib import Path

from braindecode import __version__ as braincode_version
from mne import __version__ as mne_version
import pandas as pd

from braindecode import EEGClassifier
from braindecode.datasets import create_from_mne_epochs
from braindecode.models import EEGNet, ShallowFBCSPNet, Deep4Net, EEGNetv4, ATCNet, AttentionBaseNet

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from skorch.callbacks import LRScheduler, EarlyStopping
from torch.utils.data import Subset
from skorch.helper import predefined_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
import torch
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

mne.set_log_level("ERROR")

# Set random seeds for reproducibility
seed = 42
os.environ["PYTHONHASHSEED"] = str(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MNE version: {mne_version}")
print(f"Braincode version: {braincode_version}")

In [ ]:
# Configuration of the Participant, Session and Model
PARTICIPANT_NAME = "Aliaa"
SESSION_ID = "indoor"  # (indoor, outdoor)
MODEL_NAME = "AttentionBaseNet"  # (AttentionBaseNet, EEGNet, ShallowFBCSPNet, ATCNet)

base_dir = Path.cwd()
data_dir = base_dir / "results"
assert base_dir.exists() and data_dir.exists()

# Model hyperparameters
config = {
    "model_name": MODEL_NAME,
    "n_splits": 3,

    "lr": 0.0005,
    "batch_size": 16,
    "max_epochs": 120,

    "kernel_length": 32,
    "F1": 24,
    "D": 6,
    "F2": 48,

    "dropout_rate": 0.5,
    "weight_decay": 0.005,

    "patience": 25,
    "augmentation_factor": 3,

    "filter_low": 0.5,
    "filter_high": 35.0,

    "label_smoothing": 0.15,
    "warmup_epochs": 10,
}

## Data Loading and Normalization

Load the Data, augment and normalize it using the baseline segments.

In [ ]:
def load_and_prepare_data(participant_name: str = PARTICIPANT_NAME, session: str = SESSION_ID):
    """Load preprocessed epochs for a participant and split into task and baseline.

    Parameters
    ----------
    participant_name : str
        Name of the participant (folder name under results/).
    session : str
        Session identifier (e.g. 'indoor', 'outdoor').
    """
    epochs_path = data_dir / participant_name / f"{session}_processed-epo.fif"

    if not epochs_path.exists():
        raise FileNotFoundError(f"Epochs file not found: {epochs_path}")

    epochs = mne.read_epochs(str(epochs_path), preload=True, verbose=False)

    baseline_marker = "baseline"

    if baseline_marker not in epochs.event_id:
        raise ValueError(
            f"Baseline marker '{baseline_marker}' was not found in the epochs file. "
            f"Available markers: {list(epochs.event_id.keys())}"
        )

    baseline_epochs = epochs[baseline_marker]
    task_epochs = epochs[["1-back", "2-back", "3-back"]]

    print(f"Loaded {len(task_epochs)} task epochs.")
    print(f"Found {len(baseline_epochs)} baseline epochs for normalization.")

    return task_epochs, baseline_epochs


def add_metadata_with_targets(epochs):
    """Add a metadata DataFrame with integer targets (0,1,2) mapped from n-back labels."""
    target_map = {"1-back": 0, "2-back": 1, "3-back": 2}

    try:
        event_mapping = {
            original_id: target_map[condition]
            for condition, original_id in epochs.event_id.items()
            if condition in target_map
        }
    except KeyError as e:
        raise RuntimeError(
            f"Condition '{e.args[0]}' from target_map not found in epochs.event_id. "
            f"Available conditions: {list(epochs.event_id.keys())}"
        )

    epochs_with_meta = epochs.copy()

    try:
        targets = [
            event_mapping[event_id] for event_id in epochs_with_meta.events[:, 2]
        ]
    except KeyError as e:
        raise RuntimeError(
            f"Event ID '{e.args[0]}' could not be found in the constructed mapping. "
            f"Mapping used: {event_mapping}"
        )

    # Remap event codes in-place to compact 0..N labels (required by many DL models)
    for i, original_event_id in enumerate(epochs_with_meta.events[:, 2]):
        epochs_with_meta.events[i, 2] = event_mapping[original_event_id]

    epochs_with_meta.metadata = pd.DataFrame({"target": targets})
    epochs_with_meta.event_id = target_map

    print("\nMetadata with 'target' column successfully added.")
    print("Direct mapping used (Original event ID -> Target):", event_mapping)
    print("Events were remapped to 0, 1, 2")
    print("Example metadata head:")
    print(epochs_with_meta.metadata.head())

    return epochs_with_meta


def normalize_epochs_with_baseline(task_epochs: mne.epochs.EpochsFIF,
                                   baseline_epochs: mne.epochs.EpochsFIF) -> mne.EpochsArray:
    """Z-normalize task epochs channel-wise using all baseline samples.

    Uses StandardScaler fitted on baseline (flattened per-channel samples) and applies it
    to task epochs, preserving structure.
    """
    baseline_data = baseline_epochs.get_data(copy=False)
    filtered_data = task_epochs.get_data(copy=True)

    # Reshape to (samples, channels) for StandardScaler (time collapsed across epochs)
    baseline_reshaped = baseline_data.transpose(1, 0, 2).reshape(baseline_data.shape[1], -1).T
    filtered_reshaped = filtered_data.transpose(1, 0, 2).reshape(filtered_data.shape[1], -1).T

    scaler = StandardScaler()
    scaler.fit(baseline_reshaped)

    normalized_reshaped = scaler.transform(filtered_reshaped)

    # Reshape back to (n_epochs, n_channels, n_times)
    normalized_filtered_data = normalized_reshaped.T.reshape(
        filtered_data.shape[1], filtered_data.shape[0], filtered_data.shape[2]
    ).transpose(1, 0, 2)

    normalized_epochs = mne.EpochsArray(
        normalized_filtered_data,
        task_epochs.info,
        events=task_epochs.events,
        tmin=task_epochs.tmin,
        event_id=task_epochs.event_id,
        metadata=task_epochs.metadata,
        verbose=False,
    )

    return normalized_epochs


def augment_eeg_data(epochs, augment_factor=2):
    """Apply simple data augmentation to increase training variability.

    Methods:
    1. Gaussian noise
    2. Time shifting (circular roll)
    3. Channel-wise amplitude scaling
    4. (Occasionally) channel attenuation (dropout-like)
    """
    print(f"\nAugmenting data with factor {augment_factor}")

    original_data = epochs.get_data()
    original_events = epochs.events
    original_metadata = epochs.metadata

    augmented_data_list = [original_data]
    augmented_events_list = [original_events]
    augmented_metadata_list = [original_metadata]

    for aug_idx in range(augment_factor - 1):
        augmented_data = original_data.copy()

        # Gaussian Noise
        noise_level = 0.03
        augmented_data += np.random.normal(0, noise_level, original_data.shape)

        # Time Jittering
        max_shift = 5  # samples
        for epoch_idx in range(len(augmented_data)):
            shift = np.random.randint(-max_shift, max_shift)
            if shift != 0:
                augmented_data[epoch_idx] = np.roll(augmented_data[epoch_idx], shift, axis=1)

        # Amplitude scaling per channel
        for epoch_idx in range(len(augmented_data)):
            for ch_idx in range(augmented_data.shape[1]):
                scale_factor = np.random.uniform(0.9, 1.1)  # ±10% variation
                augmented_data[epoch_idx, ch_idx] *= scale_factor

        # Channel dropout
        if np.random.random() < 0.1:  # 10% chance
            dropout_ch = np.random.randint(0, augmented_data.shape[1])
            augmented_data[:, dropout_ch] *= 0.1

        augmented_events = original_events.copy()
        augmented_events[:, 0] += len(original_data) * (aug_idx + 1)
        augmented_metadata = original_metadata.copy()

        augmented_data_list.append(augmented_data)
        augmented_events_list.append(augmented_events)
        augmented_metadata_list.append(augmented_metadata)

    combined_data = np.concatenate(augmented_data_list, axis=0)
    combined_events = np.concatenate(augmented_events_list, axis=0)
    combined_metadata = pd.concat(augmented_metadata_list, ignore_index=True)

    augmented_epochs = mne.EpochsArray(
        combined_data,
        epochs.info,
        events=combined_events,
        tmin=epochs.tmin,
        event_id=epochs.event_id,
        metadata=combined_metadata,
        verbose=False,
    )

    return augmented_epochs


sns.set_context("notebook")
sns.set_style("whitegrid")

try:
    task_epochs, baseline_epochs = load_and_prepare_data()
    task_norm = normalize_epochs_with_baseline(task_epochs, baseline_epochs)
    task_ready = add_metadata_with_targets(task_norm)
    print("Data successfully loaded and processed.")
except Exception as e:
    print(f"Error during loading or preparation: {e}")
    raise e

## Data Visualization

Quick diagnostics of class balance and a visual example of one epoch.

In [ ]:
target_counts = task_ready.metadata["target"].value_counts().sort_index()
labels = ["1-back", "2-back", "3-back"]

plt.figure(figsize=(6, 4))
ax = sns.barplot(x=labels, y=target_counts.values)
ax.set_title("Target Distribution")
ax.set_xlabel("N-Back Class")
ax.set_ylabel("Number of Epochs")
for i, v in enumerate(target_counts.values):
    ax.text(i, v + max(target_counts.values) * 0.02, str(v), ha='center')
plt.tight_layout()
plt.show()

In [ ]:
# Show an example of the first epoch (first 5 channels)
X = task_ready.get_data()
info = task_ready.info
sfreq = info["sfreq"]

if len(X) > 0:
    epoch0 = X[0]
    n_plot_ch = min(5, epoch0.shape[0])
    t = np.arange(epoch0.shape[1]) / sfreq

    plt.figure(figsize=(10, 6))
    offset = 0
    for ch in range(n_plot_ch):
        plt.plot(t, epoch0[ch] + offset, label=f"Ch {ch}")
        offset += (np.std(epoch0[ch]) * 5)
    plt.title("Example Visualization of Epoch 0 with 5 Channels")
    plt.xlabel("Time [s]")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

### Example of Augmented Data

Demonstrates the effect of the augmentation pipeline (noise, temporal jitter, per-channel scaling,
and occasional channel attenuation) on a single channel of one epoch.

In [ ]:
aug = augment_eeg_data(task_ready, augment_factor=2)
Xa = aug.get_data()

idx_epoch = 0
idx_chan = 0
sig_orig = X[idx_epoch, idx_chan]
sig_aug = Xa[len(X) + idx_epoch, idx_chan]

plt.figure(figsize=(10, 4))
plt.plot(sig_orig, alpha=0.8, label="original")
plt.plot(sig_aug, alpha=0.8, label="augmented")
plt.title("Augmentation – Example (single channel)")
plt.xlabel("Samples")
plt.ylabel("Amplitude (normalized)")
plt.legend()
plt.tight_layout()
plt.show()

### Frequency Analysis

Welch PSD per class to inspect spectral differences after preprocessing.

In [ ]:
from scipy.signal import welch

psd_dict = {}
for cls in [0, 1, 2]:
    data_cls = task_ready[task_ready.metadata["target"] == cls].get_data()
    if data_cls.size == 0:
        continue
    f, Pxx = welch(data_cls.reshape(-1, data_cls.shape[-1]), fs=sfreq, nperseg=min(256, data_cls.shape[-1]))
    psd_dict[cls] = Pxx.mean(axis=0)

plt.figure(figsize=(8, 5))
for cls, p in psd_dict.items():
    plt.semilogy(f, p, label=f"Class {cls}")
plt.title("Average PSD per Class (Welch)")
plt.xlabel("Frequency [Hz]")
plt.ylabel("PSD")
plt.legend()
plt.tight_layout()
plt.show()

## Training and Cross-Validation

Stratified K-Fold CV on normalized epochs. Each fold augments the training split, uses non-overlapping
windows, class weights, AdamW optimizer and a cosine LR schedule. We track validation accuracy and
training loss for convergence and stability.

In [ ]:
def create_cv_splits(epochs, n_splits):
    """Return a StratifiedKFold splitter preserving class balance."""
    _ = epochs.metadata["target"].values  # kept for clarity; not needed directly
    cv_splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    return cv_splitter


def get_model(model_name, n_chans, n_classes, epoch_length_s, sfreq, config):
    """Instantiate the selected model architecture with given hyperparameters."""
    if model_name == "EEGNet":
        return EEGNet(
            input_window_seconds=epoch_length_s,
            sfreq=sfreq,
            n_chans=n_chans,
            n_outputs=n_classes,
            final_conv_length="auto",
            pool_mode="mean",
            kernel_length=config["kernel_length"],
            F1=config["F1"],
            D=config["D"],
            F2=config["F2"],
            drop_prob=config["dropout_rate"],
        )
    elif model_name == "ShallowFBCSPNet":
        return ShallowFBCSPNet(
            n_chans=n_chans,
            n_outputs=n_classes,
            input_window_seconds=epoch_length_s,
            sfreq=sfreq,
            n_filters_time=40,
            n_filters_spat=40,
            final_conv_length='auto',
            drop_prob=config["dropout_rate"]
        )
    elif model_name == "ATCNet":
        return ATCNet(
            n_chans=n_chans,
            n_outputs=n_classes,
            input_window_seconds=epoch_length_s,
            sfreq=sfreq
        )
    elif model_name == "AttentionBaseNet":
        return AttentionBaseNet(
            n_chans=n_chans,
            n_outputs=n_classes,
            input_window_seconds=epoch_length_s,
            sfreq=sfreq,
        )
    else:
        raise ValueError(
            f"Unknown model: {model_name}. Available: EEGNet, ShallowFBCSPNet, ATCNet, AttentionBaseNet")


def train_eegnet_with_cv():
    """Run cross-validation training and return per-fold accuracy and confusion matrices."""
    epochs, baseline_epochs = load_and_prepare_data()
    epochs = normalize_epochs_with_baseline(epochs, baseline_epochs)
    epochs = add_metadata_with_targets(epochs)

    sfreq = epochs.info["sfreq"]
    epoch_length_s = epochs.times[-1] - epochs.times[0]
    window_size_samples = len(epochs.times)
    window_stride_samples = window_size_samples  # non-overlapping windows

    print(f"Epoch length: {epoch_length_s:.2f}s ({window_size_samples} samples)")
    print(f"Sampling frequency: {sfreq} Hz")

    n_chans = epochs.info["nchan"]
    n_classes = len(epochs.event_id)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    n_splits = config["n_splits"]
    cv_splitter = create_cv_splits(epochs, n_splits)

    all_fold_accuracies, all_fold_f1_scores = [], []
    confusion_mats = []
    fold_histories = []

    for fold, (train_idx, test_idx) in enumerate(
            cv_splitter.split(X=epochs.get_data(), y=epochs.metadata["target"])
    ):
        model = get_model(
            model_name=config["model_name"],
            n_chans=n_chans,
            n_classes=n_classes,
            epoch_length_s=epoch_length_s,
            sfreq=sfreq,
            config=config,
        )

        train_epochs = epochs[train_idx]
        test_epochs = epochs[test_idx]

        # Augment only training part
        train_epochs_augmented = augment_eeg_data(train_epochs, augment_factor=config["augmentation_factor"])

        train_dataset = create_from_mne_epochs(
            [train_epochs_augmented],
            window_size_samples=window_size_samples,
            window_stride_samples=window_stride_samples,
            drop_last_window=False,
        )
        test_dataset = create_from_mne_epochs(
            [test_epochs],
            window_size_samples=window_size_samples,
            window_stride_samples=window_stride_samples,
            drop_last_window=False,
        )

        print(
            f"Training data: {len(train_epochs_augmented)} epochs -> {len(train_dataset)} windows"
        )
        print(
            f"Validation data: {len(test_epochs)} epochs -> {len(test_dataset)} windows"
        )

        # Compute class weights to mitigate imbalance
        unique_targets = np.unique(epochs.metadata["target"])
        class_weights = compute_class_weight(
            "balanced", classes=unique_targets, y=epochs.metadata["target"]
        )
        class_weight_dict = dict(zip(unique_targets, class_weights))
        print(f"Class weights: {class_weight_dict}")

        clf = EEGClassifier(
            model,
            criterion=torch.nn.CrossEntropyLoss,
            criterion__weight=torch.FloatTensor(
                [class_weights[i] for i in range(len(class_weights))]
            ),
            criterion__label_smoothing=config["label_smoothing"],
            optimizer=torch.optim.AdamW,
            optimizer__lr=config["lr"],
            optimizer__weight_decay=config["weight_decay"],
            optimizer__betas=(0.9, 0.999),
            optimizer__eps=1e-8,
            batch_size=config["batch_size"],
            max_epochs=config["max_epochs"],
            train_split=predefined_split(test_dataset),
            device=device,
            classes=[0, 1, 2],
            callbacks=[
                (
                    "lr_scheduler",
                    LRScheduler("CosineAnnealingLR", T_max=config["max_epochs"] - 1),
                ),
            ],
            verbose=1,
        )

        print("Starting training for this fold...")
        clf.fit(train_dataset, y=None)

        # Capture per-epoch history for learning curves
        train_loss_vals = clf.history[:, 'train_loss']
        valid_loss_vals = clf.history[:, 'valid_loss'] if 'valid_loss' in clf.history[-1] else []
        valid_acc_vals = clf.history[:, 'valid_acc'] if 'valid_acc' in clf.history[-1] else []
        fold_histories.append({
            'train_loss': list(train_loss_vals),
            'valid_loss': list(valid_loss_vals) if isinstance(valid_loss_vals, (list, tuple)) else [],
            'valid_acc': list(valid_acc_vals) if isinstance(valid_acc_vals, (list, tuple)) else [],
        })

        y_true = [y for x, y, i in test_dataset]
        y_pred = clf.predict(test_dataset)

        # Prediction distribution analysis
        pred_unique, pred_counts = np.unique(y_pred, return_counts=True)
        true_unique, true_counts = np.unique(y_true, return_counts=True)

        print(f"Training finished after {len(clf.history)} epochs.")
        print(
            f"\nValidation set class distribution (ground truth): {dict(zip(true_unique, true_counts))}"
        )
        print(f"Predicted class distribution: {dict(zip(pred_unique, pred_counts))}")

        print("\nConfusion matrix:")
        print(confusion_matrix(y_true, y_pred))
        confusion_mats.append(confusion_matrix(y_true, y_pred))
        print("\nClassification report:")
        print(
            classification_report(
                y_true,
                y_pred,
                target_names=["1-back", "2-back", "3-back"],
                zero_division=0,
            )
        )

        valid_acc = clf.history[-1, "valid_acc"]
        valid_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

        all_fold_accuracies.append(valid_acc)
        all_fold_f1_scores.append(valid_f1)
        print(f"Fold {fold + 1} - Validation Accuracy: {valid_acc:.4f}, F1-Score: {valid_f1:.4f}")

    mean_accuracy = np.mean(all_fold_accuracies)
    std_accuracy = np.std(all_fold_accuracies)

    print("\n\n--- Cross-validation complete ---")
    print(
        f"Per-fold accuracies: {[f'{acc:.4f}' for acc in all_fold_accuracies]}"
    )
    print(f"Mean accuracy: {mean_accuracy:.4f}")
    print(f"Std of accuracy: {std_accuracy:.4f}")

    # Plot learning curves and confusion matrices
    try:
        sns.set_context('notebook')
        sns.set_style('whitegrid')

        # Validation Accuracy per Fold
        plt.figure(figsize=(7, 4))
        for i, h in enumerate(fold_histories, 1):
            if h.get('valid_acc'):
                plt.plot(h['valid_acc'], label=f'Fold {i}')
        plt.title('Learning Curves – Validation Accuracy (CV)')
        plt.xlabel('Epoch')
        plt.ylabel('Valid Acc')
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Training Loss per Fold
        plt.figure(figsize=(7, 4))
        for i, h in enumerate(fold_histories, 1):
            if h.get('train_loss'):
                plt.plot(h['train_loss'], label=f'Fold {i}')
        plt.title('Learning Curves – Training Loss (CV)')
        plt.xlabel('Epoch')
        plt.ylabel('Train Loss')
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Bar plot of per-fold validation accuracy with mean
        if len(all_fold_accuracies) > 0:
            arr = np.array(all_fold_accuracies, dtype=float)
            mean = arr.mean()
            plt.figure(figsize=(6, 4))
            ax = sns.barplot(x=[f'Fold {i + 1}' for i in range(len(arr))], y=arr)
            ax.axhline(mean, linestyle='--', label=f'Mean={mean:.3f}')
            ax.set_ylim(0, 1)
            ax.set_ylabel('Validation Accuracy')
            ax.set_title('Validation Accuracy per Fold')
            ax.legend()
            for i, v in enumerate(arr):
                ax.text(i, min(0.98, v + 0.02), f'{v:.3f}', ha='center')
            plt.tight_layout()
            plt.show()

        # Confusion matrices per fold
        if confusion_mats:
            for i, cm in enumerate(confusion_mats, start=1):
                plt.figure(figsize=(4.5, 4))
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                            xticklabels=['1-back', '2-back', '3-back'],
                            yticklabels=['1-back', '2-back', '3-back'])
                plt.title(f'Confusion Matrix – Fold {i}')
                plt.xlabel('Predicted')
                plt.ylabel('True')
                plt.tight_layout()
                plt.show()
    except Exception as e:
        print('Warning: Failed to plot CV visuals:', e)

    return all_fold_accuracies, confusion_mats

## Execution of the Cross-Validation

In [ ]:
results = None
try:
    results, confusion_mats = train_eegnet_with_cv()
    print("Training finished. Results (Validation Acc per Fold): %s", results)
except Exception as e:
    print(f"Error during training: {e}")
    raise e

## Final Training on all Data + Learning Curves

Train on all prepared epochs with augmentation to obtain the final model,
plot the training loss, and compute window-level metrics for a quick sanity check.
For generalization, evaluate on a held-out split or the opposite session.

In [ ]:
def train_final_model(epochs, config_override=None):
    """Train final model with augmentation and optional validation + early stopping.

    If a validation fraction > 0 is configured, the function uses a stratified
    hold-out split (on window-level labels), augments only the training part,
    and enables early stopping on 'valid_loss' to mitigate overfitting.

    Returns the trained classifier, dataset used for training, and window parameters.
    """
    # Configuration (same as above unless overridden)
    config = config_override or {
        "model_name": MODEL_NAME,
        "n_splits": 3,
        "lr": 0.0005,
        "batch_size": 16,
        "max_epochs": 120,
        "kernel_length": 32,
        "F1": 24,
        "D": 6,
        "F2": 48,
        "dropout_rate": 0.5,
        "weight_decay": 0.005,
        "patience": 25,
        "augmentation_factor": 3,
        "filter_low": 0.5,
        "filter_high": 35.0,
        "label_smoothing": 0.15,
        "warmup_epochs": 10,
        "final_valid_fraction": 0.15,  # 0 disables validation split
        "early_stopping_patience": 15,
    }

    sfreq = epochs.info["sfreq"]
    epoch_length_s = epochs.times[-1] - epochs.times[0]
    window_size_samples = len(epochs.times)
    window_stride_samples = window_size_samples

    n_chans = epochs.info["nchan"]
    n_classes = len(epochs.event_id)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = get_model(
        model_name=config["model_name"],
        n_chans=n_chans,
        n_classes=n_classes,
        epoch_length_s=epoch_length_s,
        sfreq=sfreq,
        config=config,
    )

    # Base dataset (no augmentation)
    dataset_base = create_from_mne_epochs(
        [epochs],
        window_size_samples=window_size_samples,
        window_stride_samples=window_stride_samples,
        drop_last_window=False,
    )

    # Optional validation split + early stopping
    valid_frac = float(config.get("final_valid_fraction", 0.0))
    callbacks = [("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=config["max_epochs"] - 1))]

    if valid_frac > 0.0 and len(dataset_base) > 1:
        # Labels for stratification on window level
        y_all = np.array([y for _, y, _ in dataset_base])
        idx_all = np.arange(len(dataset_base))
        sss = StratifiedShuffleSplit(n_splits=1, test_size=valid_frac, random_state=42)
        train_idx_base, val_idx_base = next(sss.split(idx_all, y_all))

        # Augment only training data (epoch-level)
        aug_factor = int(config.get("augmentation_factor", 1))
        epochs_aug = augment_eeg_data(epochs, augment_factor=aug_factor)
        dataset_aug = create_from_mne_epochs(
            [epochs_aug],
            window_size_samples=window_size_samples,
            window_stride_samples=window_stride_samples,
            drop_last_window=False,
        )
        # Map base train indices to augmented dataset indices; one window per epoch => offset by N per augmentation copy
        N = len(dataset_base)
        train_aug_idx = []
        for k in range(aug_factor):
            train_aug_idx.extend((train_idx_base + k * N).tolist())

        train_dataset = Subset(dataset_aug, train_aug_idx)
        valid_dataset = Subset(dataset_base, val_idx_base)
        split = predefined_split(valid_dataset)
        # Early stopping on validation
        callbacks.append(("early_stop",
                          EarlyStopping(patience=int(config.get("early_stopping_patience", 15)), monitor="valid_loss",
                                        lower_is_better=True, load_best=True)))
    else:
        # No validation split: train on all data with augmentation
        epochs_aug = augment_eeg_data(epochs, augment_factor=config["augmentation_factor"])
        train_dataset = create_from_mne_epochs(
            [epochs_aug],
            window_size_samples=window_size_samples,
            window_stride_samples=window_stride_samples,
            drop_last_window=False,
        )
        split = None

    # Class weights from all data
    unique_targets = np.unique(epochs.metadata["target"])
    class_weights = compute_class_weight(
        "balanced", classes=unique_targets, y=epochs.metadata["target"]
    )

    clf = EEGClassifier(
        model,
        criterion=torch.nn.CrossEntropyLoss,
        criterion__weight=torch.FloatTensor([class_weights[i] for i in range(len(class_weights))]),
        criterion__label_smoothing=config["label_smoothing"],
        optimizer=torch.optim.AdamW,
        optimizer__lr=config["lr"],
        optimizer__weight_decay=config["weight_decay"],
        optimizer__betas=(0.9, 0.999),
        optimizer__eps=1e-8,
        batch_size=config["batch_size"],
        max_epochs=config["max_epochs"],
        train_split=split,
        device=device,
        classes=[0, 1, 2],
        callbacks=callbacks,
        verbose=1,
    )

    print("\n=== Final Training (all data) ===")
    clf.fit(train_dataset, y=None)
    return clf, train_dataset, window_size_samples, window_stride_samples

In [ ]:
final_clf, final_dataset, W_final, S_final = train_final_model(task_ready)

plt.figure(figsize=(7, 4))
plt.plot(final_clf.history[:, "train_loss"])
plt.title("Final Training – Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Train Loss")
plt.tight_layout()
plt.show()

y_true_final = np.array([y for _, y, _ in final_dataset])
y_pred_final = final_clf.predict(final_dataset)
acc_final = accuracy_score(y_true_final, y_pred_final)
print(f"Final model – training accuracy (window-level): {acc_final:.4f}")
print("Confusion Matrix (Train):")
cm_train = confusion_matrix(y_true_final, y_pred_final)
print(cm_train)
print("\nClassification Report (Train):")
print(classification_report(y_true_final, y_pred_final, target_names=["1-back", "2-back", "3-back"], zero_division=0))

# Plot confusion matrices (absolute and normalized)
cmn_train = confusion_matrix(y_true_final, y_pred_final, normalize='true')
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['1-back', '2-back', '3-back'],
            yticklabels=['1-back', '2-back', '3-back'])
plt.title('Confusion (Train, abs)');
plt.xlabel('Predicted');
plt.ylabel('True')
plt.subplot(1, 2, 2)
sns.heatmap(cmn_train, annot=True, fmt='.2f', cmap='Reds', cbar=False, xticklabels=['1-back', '2-back', '3-back'],
            yticklabels=['1-back', '2-back', '3-back'], vmin=0, vmax=1)
plt.title('Confusion (Train, norm)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## Outdoor session evaluation
Evaluate the final model on the outdoor session to assess generalization to a different environment.

In [ ]:
# Check if the final model can be applied to the opposite polarity session (indoor vs. outdoor)
if SESSION_ID == "indoor":
    session_id = "outdoor"
else:
    session_id = "indoor"

try:
    polar_epochs, polar_baseline = load_and_prepare_data(participant_name=PARTICIPANT_NAME, session=session_id)
    polar_norm = normalize_epochs_with_baseline(polar_epochs, polar_baseline)
    polar_ready = add_metadata_with_targets(polar_norm)
    print("Opposite session data successfully loaded and prepared.")

    window_size_samples = len(polar_ready.times)
    window_stride_samples = window_size_samples

    polar_dataset = create_from_mne_epochs(
        [polar_ready],
        window_size_samples=window_size_samples,
        window_stride_samples=window_stride_samples,
        drop_last_window=False,
    )

    y_true_polar = np.array([y for _, y, _ in polar_dataset])
    y_pred_polar = final_clf.predict(polar_dataset)
    acc_polar = accuracy_score(y_true_polar, y_pred_polar)

    print(f"\nFinal model – accuracy on opposite session ({session_id}): {acc_polar:.4f}")
    print("Confusion Matrix (Opposite Session):")
    cm_polar = confusion_matrix(y_true_polar, y_pred_polar)
    print(cm_polar)
    print("\nClassification Report (Opposite Session):")
    print(
        classification_report(y_true_polar, y_pred_polar, target_names=["1-back", "2-back", "3-back"], zero_division=0))

    # Plot confusion matrices for opposite session
    cmn_polar = confusion_matrix(y_true_polar, y_pred_polar, normalize='true')

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.heatmap(cm_polar, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['1-back', '2-back', '3-back'],
                yticklabels=['1-back', '2-back', '3-back'])
    plt.title(f'Confusion (Opposite: {session_id}, abs)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.subplot(1, 2, 2)
    sns.heatmap(cmn_polar, annot=True, fmt='.2f', cmap='Reds', cbar=False, xticklabels=['1-back', '2-back', '3-back'],
                yticklabels=['1-back', '2-back', '3-back'], vmin=0, vmax=1)
    plt.title(f'Confusion (Opposite: {session_id}, norm)');
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Error loading/processing opposite session data: {e}")

## Channel Importance

We estimate channel importance via an occlusion analysis: zero out one channel at a time,
re-evaluate the trained classifier, and measure the drop in accuracy (ΔAcc). Larger drops
indicate higher importance. This is a sensitivity analysis, not causality.

In [ ]:
def mask_channel_epochs(epochs, ch_idx, scale=0.0):
    data = epochs.get_data().copy()
    data[:, ch_idx, :] = data[:, ch_idx, :] * scale
    return mne.EpochsArray(
        data,
        epochs.info,
        events=epochs.events,
        tmin=epochs.tmin,
        event_id=epochs.event_id,
        metadata=epochs.metadata,
        verbose=False,
    )


def channel_importance_drop(clf, base_epochs, window_size_samples, window_stride_samples):
    """Estimate channel importance by measuring accuracy drop when masking each channel."""

    base_dataset = create_from_mne_epochs(
        [base_epochs],
        window_size_samples=window_size_samples,
        window_stride_samples=window_stride_samples,
        drop_last_window=False,
    )
    y_true = np.array([y for _, y, _ in base_dataset])
    base_pred = clf.predict(base_dataset)
    base_acc = accuracy_score(y_true, base_pred)

    n_chans = base_epochs.info["nchan"]
    drops = []
    for ch in range(n_chans):
        masked = mask_channel_epochs(base_epochs, ch, scale=0.0)
        masked_dataset = create_from_mne_epochs(
            [masked],
            window_size_samples=window_size_samples,
            window_stride_samples=window_stride_samples,
            drop_last_window=False,
        )
        pred = clf.predict(masked_dataset)
        acc = accuracy_score(y_true, pred)
        drops.append(base_acc - acc)
    return base_acc, np.array(drops)

In [ ]:
# Estimate channel importance using occlusion on the (non-augmented) prepared epochs
base_acc, drops = channel_importance_drop(final_clf, task_ready, W_final, S_final)
ch_names = task_ready.info["ch_names"]

# Plot
plt.figure(figsize=(8, 4))
sns.barplot(x=ch_names, y=drops)
plt.title("Channel Importance via Occlusion (Δ Accuracy)")
plt.xlabel("Channel")
plt.ylabel("Δ Accuracy (Baseline − masked)")
plt.tight_layout()
plt.show()

# Text analysis
order = np.argsort(-drops)
print(f"Baseline accuracy (unmasked): {base_acc:.4f}")
for i in order:
    print(f"{ch_names[i]:>10}: ΔAcc = {drops[i]:+.4f}")
if len(order):
    print(f"\n→ Most influential channel: {ch_names[order[0]]} (ΔAcc={drops[order[0]]:+.4f})")

## Train-Test Split

The next section trains with a simple  train/test split on a single session (no K-Fold). This complements the main CV-based analysis and mitigates data leakage risks.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import mne
import torch
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from braindecode import EEGClassifier
from braindecode.datasets import create_from_mne_epochs
from skorch.helper import predefined_split
from skorch.callbacks import LRScheduler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

mne.set_log_level('ERROR')

config = {
    'subject': PARTICIPANT_NAME,
    'file_type': SESSION_ID,
    'segment_length': 4.0,
    'overlap_train': 2.0,
    'lr': 5e-4,
    'batch_size': 16,
    'max_epochs': 20,
    'dropout': 0.35,
    'weight_decay': 0.01,
    'label_smoothing': 0.1,
    'use_balanced_class_weights': True,
    'results_dir': Path.cwd() / "results",
}


def create_simple_train_test_split(raw):
    task_groups = {'1-back': [], '2-back': [], '3-back': []}
    baseline_indices = []
    for i, annot in enumerate(raw.annotations):
        desc = annot['description']
        if desc in task_groups:
            task_groups[desc].append((i, annot['onset']))
        elif desc == 'baseline':
            baseline_indices.append(i)
    train_indices, test_indices = [], []
    for task_type, annotations in task_groups.items():
        if len(annotations) != 2:
            raise ValueError(f"Expected exactly 2 phases for {task_type}, found: {len(annotations)}")
        annotations_sorted = sorted(annotations, key=lambda x: x[1])
        train_indices.append(annotations_sorted[0][0])
        test_indices.append(annotations_sorted[1][0])
    train_indices.extend(baseline_indices)
    return sorted(train_indices), sorted(test_indices), baseline_indices


def create_epochs_for_split(raw, annotation_indices, use_overlap=True, segment_length=4.0, overlap=2.0):
    temp_annotations = mne.Annotations(
        onset=[raw.annotations.onset[i] for i in annotation_indices],
        duration=[raw.annotations.duration[i] for i in annotation_indices],
        description=[raw.annotations.description[i] for i in annotation_indices],
        orig_time=raw.annotations.orig_time
    )
    temp_raw = raw.copy().set_annotations(temp_annotations)
    actual_overlap = overlap if use_overlap else 0.0
    base_event_id = {'baseline': 0, '1-back': 1, '2-back': 2, '3-back': 3}
    present = {d for d in temp_raw.annotations.description}
    event_id = {k: v for k, v in base_event_id.items() if k in present}
    all_events = []
    for annot in temp_raw.annotations:
        d = annot['description']
        if d not in event_id:
            continue
        ev = mne.make_fixed_length_events(
            temp_raw,
            id=event_id[d],
            start=annot['onset'],
            stop=annot['onset'] + annot['duration'],
            duration=segment_length,
            overlap=actual_overlap
        )
        if len(ev):
            all_events.append(ev)
    if not all_events:
        return None
    events = np.vstack(all_events)
    events = events[events[:, 0].argsort()]
    epochs = mne.Epochs(temp_raw, events=events, event_id=event_id,
                        tmin=0.0, tmax=segment_length, baseline=None,
                        preload=True, verbose=False)
    return epochs


def add_metadata_with_targets(epochs):
    target_map = {"1-back": 0, "2-back": 1, "3-back": 2}
    avail = [c for c in target_map if c in epochs.event_id]
    task_epochs = epochs[avail]
    mapping = {orig: target_map[name] for name, orig in task_epochs.event_id.items()}
    targets = [mapping[eid] for eid in task_epochs.events[:, 2]]
    for i, eid in enumerate(task_epochs.events[:, 2]):
        task_epochs.events[i, 2] = mapping[eid]
    task_epochs.metadata = pd.DataFrame({'target': targets})
    task_epochs.event_id = target_map
    return task_epochs


def normalize_epochs_with_baseline(task_epochs, baseline_epochs):
    base = baseline_epochs.get_data()
    task = task_epochs.get_data().copy()
    base_flat = base.transpose(1, 0, 2).reshape(base.shape[1], -1).T
    task_flat = task.transpose(1, 0, 2).reshape(task.shape[1], -1).T
    scaler = StandardScaler().fit(base_flat)
    task_norm_flat = scaler.transform(task_flat)
    task_norm = task_norm_flat.T.reshape(task.shape[1], task.shape[0], task.shape[2]).transpose(1, 0, 2)
    norm_epochs = mne.EpochsArray(task_norm, task_epochs.info, events=task_epochs.events,
                                  tmin=task_epochs.tmin, event_id=task_epochs.event_id,
                                  metadata=task_epochs.metadata, verbose=False)
    return norm_epochs


def create_detailed_confusion_matrix(y_true, y_pred, class_names=None, save_path=None):
    if class_names is None:
        class_names = ['1-back', '2-back', '3-back']
    cm = confusion_matrix(y_true, y_pred)
    cmn = confusion_matrix(y_true, y_pred, normalize='true')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    axes[0].set_title('Confusion (abs)')
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Reds', ax=axes[1], cbar=False,
                xticklabels=class_names, yticklabels=class_names, vmin=0, vmax=1)
    axes[1].set_title('Confusion (norm)')
    for ax in axes:
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=250, bbox_inches='tight')
    plt.show()
    return cm, cmn


raw_path = config['results_dir'] / config['subject'] / f"{config['file_type']}_processed_raw.fif"
raw = mne.io.read_raw_fif(str(raw_path), preload=True, verbose=False)

train_indices, test_indices, baseline_indices = create_simple_train_test_split(raw)

baseline_epochs = create_epochs_for_split(
    raw, baseline_indices, use_overlap=False,
    segment_length=config['segment_length'], overlap=0.0
)

train_epochs = create_epochs_for_split(
    raw, train_indices, use_overlap=True,
    segment_length=config['segment_length'], overlap=config['overlap_train']
)

test_epochs = create_epochs_for_split(
    raw, test_indices, use_overlap=False,
    segment_length=config['segment_length'], overlap=0.0
)

if train_epochs is None or test_epochs is None:
    raise RuntimeError("Error during epoch creation.")

train_epochs = normalize_epochs_with_baseline(train_epochs, baseline_epochs)
test_epochs = normalize_epochs_with_baseline(test_epochs, baseline_epochs)

train_epochs = add_metadata_with_targets(train_epochs)
test_epochs = add_metadata_with_targets(test_epochs)

win_samples = len(train_epochs.times)
train_ds = create_from_mne_epochs([train_epochs], window_size_samples=win_samples,
                                  window_stride_samples=win_samples, drop_last_window=False)

test_ds = create_from_mne_epochs([test_epochs], window_size_samples=win_samples,
                                 window_stride_samples=win_samples, drop_last_window=False)

train_counts = train_epochs.metadata['target'].value_counts().sort_index()
print("Distribution (Train):", dict(train_counts))
labels = np.array(sorted(train_counts.index.values))
if config['use_balanced_class_weights']:
    class_weights = compute_class_weight('balanced', classes=labels, y=train_epochs.metadata['target'])
else:
    class_weights = np.ones_like(labels, dtype=float)
print("Class weights:", class_weights)

sfreq = train_epochs.info['sfreq']
model = get_model(
    model_name="AttentionBaseNet",
    n_chans=8,
    n_classes=3,
    epoch_length_s=4,
    sfreq=sfreq,
    config=config,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

criterion_params = {
    'weight': torch.tensor(class_weights, dtype=torch.float32),
    'label_smoothing': config['label_smoothing']
}

clf = EEGClassifier(
    model,
    criterion=torch.nn.CrossEntropyLoss,
    **{f"criterion__{k}": v for k, v in criterion_params.items()},
    optimizer=torch.optim.AdamW,
    optimizer__lr=config['lr'],
    optimizer__weight_decay=config['weight_decay'],
    batch_size=config['batch_size'],
    max_epochs=config['max_epochs'],
    train_split=predefined_split(test_ds),
    device=device,
    callbacks=[('lr_scheduler', LRScheduler('CosineAnnealingLR', T_max=config['max_epochs'] - 1))],
    classes=[0, 1, 2],
    verbose=1
)

clf.fit(train_ds, y=None)

y_true = [y for X, y, i in test_ds]
y_pred = clf.predict(test_ds)
acc = (np.array(y_true) == np.array(y_pred)).mean()

cm_path = config['results_dir'] / 'confusion_matrix_simple_split.png'
create_detailed_confusion_matrix(y_true, y_pred, class_names=['1-back', '2-back', '3-back'], save_path=cm_path)
print("Classification Report:\n",
      classification_report(y_true, y_pred, target_names=['1-back', '2-back', '3-back'], zero_division=0))
print(f"Saved: {cm_path}")

## Channel Importance for Simple Split

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("notebook")
sns.set_style("whitegrid")

try:
    base_acc_simple, drops_simple = channel_importance_drop(
        clf, test_epochs, win_samples, win_samples
    )
    ch_names_simple = test_epochs.info['ch_names']

    plt.figure(figsize=(8, 4))
    sns.barplot(x=ch_names_simple, y=drops_simple)
    plt.title("Simple Split – Channel Importance via Occlusion (Δ Accuracy)")
    plt.xlabel("Channel")
    plt.ylabel("Δ Accuracy (Baseline − masked)")
    plt.tight_layout()
    plt.show()

    order = np.argsort(-drops_simple)
except Exception as e:
    print("Error during channel importance estimation:", e)